# One SAP → AE TFL Shell | End-to-End MVP

**Only one upload:** the SAP PDF. This notebook ONLY calls code in the GitHub `tfl_agent` package. No intermediary ZIPs, API keys, or human review required to produce an **unapproved structural draft**.

Pilot scope: Overall AE Table based on a recognizable Section 8.8.2 (optimized for RP6530-2101 / SAP_001_00.pdf). The output is not a clinically validated shell.

## Step 0 — Clone and install the official GitHub Agent package

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'feature/official-library-colab'
PROJECT_ROOT = Path('/content/tfl-automatic-ae-pilot')
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', '-B', BRANCH, 'FETCH_HEAD'])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError('PROJECT_ROOT exists but is not a Git repository')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('GitHub package:', subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## Step 1 — Upload one SAP PDF

Upload one **text-based PDF**, not a retrieval ZIP, adaptation plan, template or JSON file.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one SAP PDF')
file_name = next(iter(uploaded))
if not file_name.lower().endswith('.pdf'):
    raise ValueError('Expected a SAP PDF')
SAP_PATH = (Path.cwd() / file_name).resolve()
if not SAP_PATH.is_file():
    SAP_PATH = (Path('/content') / file_name).resolve()
assert SAP_PATH.is_file(), f'Uploaded file missing: {file_name}'
print('Loaded SAP:', SAP_PATH.name)

## Step 2 — Run the complete Agent pipeline (one call)

All extraction and structural DOCX generation are **GitHub package implementations**, not notebook cells:

SAP parsing → AE specification → official keyword template retrieval → diagnostic adaptation plan → structural Word rebuild → validation. No manual adaptation-plan approval is needed to generate the **unapproved** draft.

In [ ]:
from tfl_agent.agent.end_to_end_ae_pilot import run_single_sap_ae_pilot
OUTPUT_DIR = Path('/content/AE_TFL_End_to_End_Outputs')
result = run_single_sap_ae_pilot(
    sap_path=SAP_PATH,
    library_path=PROJECT_ROOT / 'library',
    output_dir=OUTPUT_DIR,
)
print(json.dumps({
    'Status': result['status'],
    'Target': result['target_id'],
    'Official Library count': result['approved_library_count'],
    'Selected style template': result['chosen_style_template_id'],
    'Target AE rows': result['generation']['row_count'],
    'Word columns': result['generation']['column_count'],
    'Structural validation': result['validation']['valid_structural_shell'],
    'Clinical approval': result['validation']['clinical_validity_verified'],
}, indent=2))
assert result['validation']['valid_structural_shell'], result['validation']['structural_errors']

## Step 3 — Inspect and download outputs

Editable Word structural **draft**, extraction evidence, Specification, Retrieval, Adaptation Plans and validation files. Keep the ZIP private if the SAP is confidential.

In [ ]:
from IPython.display import display, FileLink
from google.colab import files
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)
    if path.suffix == '.docx':
        display(FileLink(str(path), result_html_prefix='Editable AE draft: '))
archive = Path(shutil.make_archive(
    '/content/AE_TFL_End_to_End_Outputs', 'zip', root_dir=OUTPUT_DIR
))
files.download(str(archive))

**MVP limits:** Works only for a recognizable Overall AE SAP section and an explicit two-group design. No patient data are estimated; numeric cells remain `xx`. Footnote applicability, denominator and formatting need clinical review. The result is a structural shell **draft**, not an approved clinical TFL.